In [1]:
%pip install -q pandas huggingface_hub

In [2]:
import pandas as pd
import requests
from datasets import load_dataset
from tqdm.auto import tqdm
import time

# (test) Publication Year Metadata Validation

## Fetch Estimated Publication Years from the Project Gutenberg Temporal Corpus

Source: Momen, Schaaf, and Mehler (2025), *Filling the Temporal Void: Recovering Missing Publication Years in the Project Gutenberg Corpus Using LLMs*.

Dataset: Texttechnologylab/project-gutenberg-temporal-corpus on Hugging Face.

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "Texttechnologylab/project-gutenberg-temporal-corpus",
    split="all",
    streaming=True,
)

print(ds.column_names)

README.md:   0%|          | 0.00/2.80k [00:00<?, ?B/s]

None


In [ ]:
first_row = next(iter(ds))

print(first_row)
print(first_row.keys())

{'Book_ID': 1, 'Book_Title': 'The Declaration of Independence of the United States of America', 'Base_llama': 1776, 'LLM_web': 1776, 'RAG_wikipedia': 1776, 'RAG_book': 1776, 'LLM_first': 1789, 'LLM_first_last_gemma': 1776, 'LLM_first_last_summary': 1776, 'LLM_detailed_prompt': 1776, 'LLM_lifespan_prompt': 1776, 'LLM_first_last_llama': 1776, 'LLM_first_last_mistral': 1776, 'LLM_first_last_wizard': 1776, 'LLM_first_last_llama_full_precision': 1776, 'LLM_first_last_llama_70B': 1776, 'Open_Library': 2009}
dict_keys(['Book_ID', 'Book_Title', 'Base_llama', 'LLM_web', 'RAG_wikipedia', 'RAG_book', 'LLM_first', 'LLM_first_last_gemma', 'LLM_first_last_summary', 'LLM_detailed_prompt', 'LLM_lifespan_prompt', 'LLM_first_last_llama', 'LLM_first_last_mistral', 'LLM_first_last_wizard', 'LLM_first_last_llama_full_precision', 'LLM_first_last_llama_70B', 'Open_Library'])


In [ ]:
wanted_ids = {
1023,
9296,
158,
6053,
84,
1400,
786,
1260,
946,
141,
2153,
145,
6688,
370,
71865,
3268,
121,
730,
883,
6124,
580,
42671,
521,
30344,
30486,
6593,
599,
9182,
768,
}

In [ ]:
rows = []

for row in ds:
    book_id = int(row["Book_ID"])

    if book_id in wanted_ids:
        rows.append(row)
        print(f"Found {book_id}: {row['Book_Title']}")

    if len(rows) == len(wanted_ids):
        break

result = pd.DataFrame(rows)

print(f"\nFound {len(result)} of {len(wanted_ids)} books.")
display(result)

Found 84: Frankenstein; Or  The Modern Prometheus
Found 121: Northanger Abbey
Found 141: Mansfield Park
Found 145: Middlemarch
Found 158: Emma
Found 370: The Fortunes and Misfortunes of the Famous Moll Flanders
Found 521: The Life and Adventures of Robinson Crusoe
Found 580: The Pickwick Papers
Found 599: Vanity Fair
Found 730: Oliver Twist
Found 768: Wuthering Heights
Found 786: Hard Times
Found 883: Our Mutual Friend
Found 946: Lady Susan
Found 1023: Bleak House
Found 1260: Jane Eyre: An Autobiography
Found 1400: Great Expectations
Found 2153: Mary Barton
Found 3268: The Mysteries of Udolpho
Found 6053: Evelina  Or  the History of a Young Lady's Entrance into the World
Found 6124: Pamela  or Virtue Rewarded
Found 6593: History of Tom Jones  a Foundling
Found 6688: The Mill on the Floss
Found 9182: Villette
Found 9296: Clarissa Harlowe; or the history of a young lady — Volume 1
Found 30344: The Fortunate Mistress (Parts 1 and 2) -  - or a History of the Life of Mademoiselle de Beleau 

,Book_ID,Book_Title,Base_llama,LLM_web,RAG_wikipedia,RAG_book,LLM_first,LLM_first_last_gemma,LLM_first_last_summary,LLM_detailed_prompt,LLM_lifespan_prompt,LLM_first_last_llama,LLM_first_last_mistral,LLM_first_last_wizard,LLM_first_last_llama_full_precision,LLM_first_last_llama_70B,Open_Library
0,84,Frankenstein; Or The Modern Prometheus,1818,1818,1818,1818,1818,1818,1818,1818,1818,1818,1818,1818,1818,1818,1718
1,121,Northanger Abbey,1817,1818,1817,1818,1803,1818,1818,1803,1803,1818,1818,1818,1818,1803,1818
2,141,Mansfield Park,1814,1814,1814,1814,1814,1814,1814,1814,1814,1814,1814,1814,1814,1814,1814
3,145,Middlemarch,1871,1871,1871,1871,1871,1871,1871,1871,1871,1871,1871,1872,1871,1871,1871
4,158,Emma,1816,1816,1815,1816,1816,1815,1815,1816,1816,1816,1813,1815,1816,1816,1789
5,370,The Fortunes and Misfortunes of the Famous Mol...,1683,1722,1722,1683,1696,1722,1722,1683,1683,1683,1722,1722,1683,1683,1722
6,521,The Life and Adventures of Robinson Crusoe,1719,1840,1719,1669,1651,1719,1719,1651,1719,1669,1719,1719,1669,1694,1719
7,580,The Pickwick Papers,1836,1837,1836,1836,1827,1836,1836,1836,1836,1836,1836,1836,1836,1870,1800
8,599,Vanity Fair,1847,1848,1848,1847,1848,1848,1848,1848,1848,1848,1847,1848,1848,1848,1800
9,730,Oliver Twist,1838,1837,1838,1870,1838,1837,1838,1838,1846,1846,1838,1837,1838,1841,1822


## Load Manually Collected Wikipedia Publication Dates

In [ ]:
wikipedia_date_df = pd.read_csv('/content/Armstrong_index_books - with publication year.csv')
wikipedia_date_df

,gutenberg_id,index_title,index_author,year,gutenberg_title,gutenberg_author,txt_url,category,notes,narrative_mode,wikipedia year
0,1023,Bleak House,Dickens,NaN,Bleak House,"Dickens, Charles",https://www.gutenberg.org/ebooks/1023.txt.utf-8,NaN,NaN,mixed,1852 serialised\n1853 published
1,9296,Clarissa,Richardson,NaN,Clarissa Harlowe; or the history of a young la...,"Richardson, Samuel",https://www.gutenberg.org/ebooks/9296.txt.utf-8,core_domestic,NaN,epistolary,The first two volumes were published in Decemb...
2,158,Emma,Austen,NaN,Emma,"Austen, Jane",https://www.gutenberg.org/ebooks/158.txt.utf-8,core_domestic,most central,third_person,"published on 23 December 1815, although the ti..."
3,6053,Evelina,Burney,NaN,"Evelina, Or, the History of a Young Lady's Ent...","Burney, Fanny",https://www.gutenberg.org/ebooks/6053.txt.utf-8,NaN,NaN,epistolary,1778
4,84,Frankenstein,Shelley,NaN,"Frankenstein; Or, The Modern Prometheus","Shelley, Mary Wollstonecraft",https://www.gutenberg.org/files/84/84-0.txt,NaN,NaN,mixed,1818
5,1400,Great Expectations,Dickens,NaN,Great Expectations,"Dickens, Charles",https://www.gutenberg.org/files/1400/1400-0.txt,NaN,NaN,first_person,serialised 1860. book form 1861
6,786,Hard Times,Dickens,NaN,Hard Times,"Dickens, Charles",https://www.gutenberg.org/files/786/786-0.txt,NaN,NaN,third_person,Serialised April 1854 – 12 August 1854...
7,1260,Jane Eyre,Brontë,NaN,Jane Eyre: An Autobiography,"Brontë, Charlotte",https://www.gutenberg.org/files/1260/1260-0.txt,core_domestic,most central,third_person,Published on 19 October 1847 under the pen nam...
8,946,Lady Susan,Austen,NaN,Lady Susan,"Austen, Jane",https://www.gutenberg.org/ebooks/946.txt.utf-8,core_domestic,NaN,epistolary,1871
9,141,Mansfield Park,Austen,NaN,Mansfield Park,"Austen, Jane",https://www.gutenberg.org/files/141/141-0.txt,core_domestic,NaN,third_person,Published on 2 July 1814


## 3. Merge Records by Project Gutenberg ID

In [ ]:
merged_df = pd.merge(result, wikipedia_date_df, left_on = 'Book_ID', right_on='gutenberg_id', how='inner')
merged_df.head()

,Book_ID,Book_Title,Base_llama,LLM_web,RAG_wikipedia,RAG_book,LLM_first,LLM_first_last_gemma,LLM_first_last_summary,LLM_detailed_prompt,...,index_title,index_author,year,gutenberg_title,gutenberg_author,txt_url,category,notes,narrative_mode,wikipedia year
0,84,Frankenstein; Or The Modern Prometheus,1818,1818,1818,1818,1818,1818,1818,1818,...,Frankenstein,Shelley,NaN,"Frankenstein; Or, The Modern Prometheus","Shelley, Mary Wollstonecraft",https://www.gutenberg.org/files/84/84-0.txt,NaN,NaN,mixed,1818
1,121,Northanger Abbey,1817,1818,1817,1818,1803,1818,1818,1803,...,Northanger Abbey,Austen,NaN,Northanger Abbey,"Austen, Jane",https://www.gutenberg.org/ebooks/121.txt.utf-8,NaN,NaN,third_person,"Published on 20 December 1817, although the ti..."
2,141,Mansfield Park,1814,1814,1814,1814,1814,1814,1814,1814,...,Mansfield Park,Austen,NaN,Mansfield Park,"Austen, Jane",https://www.gutenberg.org/files/141/141-0.txt,core_domestic,NaN,third_person,Published on 2 July 1814
3,145,Middlemarch,1871,1871,1871,1871,1871,1871,1871,1871,...,Middlemarch,Eliot,NaN,Middlemarch,"Eliot, George",https://www.gutenberg.org/ebooks/145.txt.utf-8,core_domestic,NaN,third_person,1871-1872
4,158,Emma,1816,1816,1815,1816,1816,1815,1815,1816,...,Emma,Austen,NaN,Emma,"Austen, Jane",https://www.gutenberg.org/ebooks/158.txt.utf-8,core_domestic,most central,third_person,"published on 23 December 1815, although the ti..."


In [ ]:
merged_df.columns

Index(['Book_ID', 'Book_Title', 'Base_llama', 'LLM_web', 'RAG_wikipedia',
       'RAG_book', 'LLM_first', 'LLM_first_last_gemma',
       'LLM_first_last_summary', 'LLM_detailed_prompt', 'LLM_lifespan_prompt',
       'LLM_first_last_llama', 'LLM_first_last_mistral',
       'LLM_first_last_wizard', 'LLM_first_last_llama_full_precision',
       'LLM_first_last_llama_70B', 'Open_Library', 'gutenberg_id',
       'index_title', 'index_author', 'year', 'gutenberg_title',
       'gutenberg_author', 'txt_url', 'category', 'notes', 'narrative_mode',
       'wikipedia year'],
      dtype='object')

In [ ]:
comparison_df = merged_df[['Book_ID', 'Book_Title', 'gutenberg_id',
       'index_title', 'index_author', 'gutenberg_title',
       'gutenberg_author',
       'wikipedia year', 'LLM_first_last_gemma', 'RAG_wikipedia', 'RAG_book']]
comparison_df

,Book_ID,Book_Title,gutenberg_id,index_title,index_author,gutenberg_title,gutenberg_author,wikipedia year,LLM_first_last_gemma,RAG_wikipedia,RAG_book
0,84,Frankenstein; Or The Modern Prometheus,84,Frankenstein,Shelley,"Frankenstein; Or, The Modern Prometheus","Shelley, Mary Wollstonecraft",1818,1818,1818,1818
1,121,Northanger Abbey,121,Northanger Abbey,Austen,Northanger Abbey,"Austen, Jane","Published on 20 December 1817, although the ti...",1818,1817,1818
2,141,Mansfield Park,141,Mansfield Park,Austen,Mansfield Park,"Austen, Jane",Published on 2 July 1814,1814,1814,1814
3,145,Middlemarch,145,Middlemarch,Eliot,Middlemarch,"Eliot, George",1871-1872,1871,1871,1871
4,158,Emma,158,Emma,Austen,Emma,"Austen, Jane","published on 23 December 1815, although the ti...",1815,1815,1816
5,370,The Fortunes and Misfortunes of the Famous Mol...,370,Moll Flanders,Defoe,The Fortunes and Misfortunes of the Famous Mol...,"Defoe, Daniel",1722,1722,1722,1683
6,521,The Life and Adventures of Robinson Crusoe,521,Robinson Crusoe,Defoe,The Life and Adventures of Robinson Crusoe,"Defoe, Daniel",25 April 1719,1719,1719,1669
7,580,The Pickwick Papers,580,Pickwick Papers,Dickens,The Pickwick Papers,"Dickens, Charles",Serialised March 1836 – November 1837; book fo...,1836,1836,1836
8,599,Vanity Fair,599,Vanity Fair,Thackeray,Vanity Fair,"Thackeray, William Makepeace",January 1847 to July 1848,1848,1848,1847
9,730,Oliver Twist,730,Oliver Twist,Dickens,Oliver Twist,"Dickens, Charles",Serialised 1837–1839; book form 1838,1837,1838,1870


In [ ]:
output_df = merged_df[['gutenberg_id',
       'index_title', 'index_author', 'year', 'gutenberg_title',
       'gutenberg_author', 'txt_url', 'category', 'notes', 'narrative_mode',
       'wikipedia year', 'LLM_first_last_gemma', 'RAG_wikipedia']]

In [ ]:
output_df.to_csv('with_publication_year.csv', index=False)

# (test) gender & nationality

In [ ]:
wanted_ids = output_df["gutenberg_id"].dropna().astype(int).tolist()

id_values = " ".join(f'"{book_id}"' for book_id in wanted_ids)

query = f"""
SELECT DISTINCT
  ?gutenberg_id
  ?authorLabel
  ?genderLabel
  ?citizenshipLabel
  ?authorDescription
WHERE {{
  VALUES ?gutenberg_id {{
    {id_values}
  }}

  # P2034 = Project Gutenberg ebook ID
  ?work wdt:P2034 ?gutenberg_id.

  # P50 = author
  ?work wdt:P50 ?author.

  # P21 = gender
  OPTIONAL {{
    ?author wdt:P21 ?gender.
  }}

  # P27 = citizenship
  OPTIONAL {{
    ?author wdt:P27 ?citizenship.
  }}

  # English description
  OPTIONAL {{
    ?author schema:description ?authorDescription.
    FILTER(LANG(?authorDescription) = "en")
  }}

  SERVICE wikibase:label {{
    bd:serviceParam wikibase:language "en".
  }}
}}
ORDER BY ?gutenberg_id
"""

In [ ]:
print(query)


SELECT DISTINCT
  ?gutenberg_id
  ?authorLabel
  ?genderLabel
  ?citizenshipLabel
  ?authorDescription
WHERE {
  VALUES ?gutenberg_id {
    "84" "121" "141" "145" "158" "370" "521" "580" "599" "730" "768" "786" "883" "946" "1023" "1260" "1400" "2153" "3268" "6053" "6124" "6593" "6688" "9182" "9296" "30344" "30486" "42671" "71865"
  }

  # P2034 = Project Gutenberg ebook ID
  ?work wdt:P2034 ?gutenberg_id.

  # P50 = author
  ?work wdt:P50 ?author.

  # P21 = gender
  OPTIONAL {
    ?author wdt:P21 ?gender.
  }

  # P27 = citizenship
  OPTIONAL {
    ?author wdt:P27 ?citizenship.
  }

  # English description
  OPTIONAL {
    ?author schema:description ?authorDescription.
    FILTER(LANG(?authorDescription) = "en")
  }

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "en".
  }
}
ORDER BY ?gutenberg_id



In [ ]:
response = requests.get(
    "https://query.wikidata.org/sparql",
    params={
        "query": query,
        "format": "json",
    },
    headers={
        "User-Agent": "InteriorityResearch/1.0"
    },
    timeout=60,
)

response.raise_for_status()

test_data = response.json()

In [ ]:
rows = []

for item in test_data["results"]["bindings"]:
    rows.append({
        "gutenberg_id": item["gutenberg_id"]["value"],
        "author_label": item.get("authorLabel", {}).get("value"),
        "gender": item.get("genderLabel", {}).get("value"),
        "citizenship": item.get("citizenshipLabel", {}).get("value"),
        "author_description": item.get(
            "authorDescription", {}
        ).get("value"),
    })

wikidata_df_test = pd.DataFrame(rows)

display(wikidata_df_test)

,gutenberg_id,author_label,gender,citizenship,author_description
0,1023,Charles Dickens,male,United Kingdom of Great Britain and Ireland,English novelist and social critic (1812–1870)
1,121,Jane Austen,female,United Kingdom of Great Britain and Ireland,English novelist (1775–1817)
2,1260,Charlotte Brontë,female,United Kingdom of Great Britain and Ireland,British novelist and poet (1816-1855)
3,1400,Charles Dickens,male,United Kingdom of Great Britain and Ireland,English novelist and social critic (1812–1870)
4,141,Jane Austen,female,United Kingdom of Great Britain and Ireland,English novelist (1775–1817)
5,145,George Eliot,female,United Kingdom of Great Britain and Ireland,"English novelist, essayist, poet and journalis..."
6,158,Jane Austen,female,United Kingdom of Great Britain and Ireland,English novelist (1775–1817)
7,2153,Elizabeth Gaskell,female,United Kingdom of Great Britain and Ireland,British author (1810-1865)
8,30344,Daniel Defoe,male,Kingdom of England,"English trader, writer, and journalist (1660–1..."
9,30344,Daniel Defoe,male,Kingdom of Great Britain,"English trader, writer, and journalist (1660–1..."


In [ ]:
print(
    "Unique books matched:",
    wikidata_df_test["gutenberg_id"].nunique(),
    "/",
    len(wanted_ids),
)

Unique books matched: 21 / 29


In [25]:
def combine_unique(series):
    values = sorted({
        str(value).strip()
        for value in series.dropna()
        if str(value).strip()
    })

    return ", ".join(values) if values else pd.NA


wikidata_clean_df = (
    wikidata_df_test
    .groupby(
        [
            "gutenberg_id",
            "author_label",
        ],
        as_index=False,
    )
    .agg(
        gender=("gender", combine_unique),
        citizenship=("citizenship", combine_unique),
        author_description=("author_description", combine_unique),
    )
)

display(wikidata_clean_df)

,gutenberg_id,author_label,gender,citizenship,author_description
0,1023,Charles Dickens,male,United Kingdom of Great Britain and Ireland,English novelist and social critic (1812–1870)
1,121,Jane Austen,female,United Kingdom of Great Britain and Ireland,English novelist (1775–1817)
2,1260,Charlotte Brontë,female,United Kingdom of Great Britain and Ireland,British novelist and poet (1816-1855)
3,1400,Charles Dickens,male,United Kingdom of Great Britain and Ireland,English novelist and social critic (1812–1870)
4,141,Jane Austen,female,United Kingdom of Great Britain and Ireland,English novelist (1775–1817)
5,145,George Eliot,female,United Kingdom of Great Britain and Ireland,"English novelist, essayist, poet and journalis..."
6,158,Jane Austen,female,United Kingdom of Great Britain and Ireland,English novelist (1775–1817)
7,2153,Elizabeth Gaskell,female,United Kingdom of Great Britain and Ireland,British author (1810-1865)
8,30344,Daniel Defoe,male,"Kingdom of England, Kingdom of Great Britain","English trader, writer, and journalist (1660–1..."
9,30486,Charlotte Brontë,female,United Kingdom of Great Britain and Ireland,British novelist and poet (1816-1855)


In [26]:
print(wikidata_clean_df["gutenberg_id"].nunique())

21


In [27]:
matched_ids = set(
    wikidata_clean_df["gutenberg_id"]
    .astype(int)
)

missing_ids = sorted(
    set(wanted_ids) - matched_ids
)

print(missing_ids)

[521, 730, 768, 6593, 6688, 9296, 42671, 71865]


# pg_candidate_v1

### test the script:

In [28]:
%run "/content/get_book_metadata_test100.py"

Testing 100 rows
Unique Gutenberg IDs: 100
Unique author names: 90


Finding publication years: 0it [00:00, ?it/s]

Publication years found: 91 / 100
Authors matched by Gutenberg ID: 9 / 100


Matching remaining authors by name:   0%|          | 0/82 [00:00<?, ?it/s]

Author metadata found: 21 / 100

Saved main test result: /content/pg_candidate_test100_with_metadata.csv
Saved author match report: /content/pg_candidate_test100_author_match_report.csv
Final shape: (100, 22)
New columns: ['publication_year_llm_first_last_gemma', 'author_gender', 'author_citizenship', 'author_description']

Missing values in the four new columns:
publication_year_llm_first_last_gemma     9
author_gender                            80
author_citizenship                       80
author_description                       79
dtype: int64


## get author metatdata using Gutenberg ID

In [ ]:
# import time

# import pandas as pd
# import requests
# from datasets import load_dataset
# from tqdm.auto import tqdm

In [ ]:
temporal_ds = load_dataset(
    "Texttechnologylab/project-gutenberg-temporal-corpus",
    split="all",
    streaming=True,
)

In [30]:
INPUT_CSV = "/content/pg_candidate_v1 - pg_candidate_v1.csv"

books_df = pd.read_csv(INPUT_CSV)

# Make sure Gutenberg IDs are numeric.
books_df["gutenberg_id"] = pd.to_numeric(
    books_df["gutenberg_id"],
    errors="coerce"
).astype("Int64")

# Remove rows without a usable Gutenberg ID.
books_df = books_df.loc[
    books_df["gutenberg_id"].notna()
].copy()

print("Number of books:", len(books_df))
print(
    "Unique Gutenberg IDs:",
    books_df["gutenberg_id"].nunique()
)

books_df.head()

Number of books: 7307
Unique Gutenberg IDs: 7307


,gutenberg_id,title,subjects,subject_text,author_raw,author_display,author_query,is_domestic,is_detective_mystery,is_science_fiction,is_adventure,is_gothic,is_western,is_fantasy,is_historical,has_major_genre,genre_tags,broad_group
0,24,O Pioneers!,Nebraska -- Fiction; Historical fiction; Front...,nebraska -- fiction; historical fiction; front...,"Cather, Willa, 1873-1947",Willa Cather,Willa Cather,1,0,0,0,0,0,0,1,1,historical,domestic_and_genre
1,46,A Christmas Carol in Prose; Being a Ghost Stor...,Christmas stories; London (England) -- Fiction...,christmas stories; london (england) -- fiction...,"Dickens, Charles, 1812-1870; Leech, John, 1817...",Charles Dickens,Charles Dickens,1,0,0,0,1,0,0,0,1,gothic,domestic_and_genre
2,77,The House of the Seven Gables,Historical fiction; Domestic fiction; Paranorm...,historical fiction; domestic fiction; paranorm...,"Hawthorne, Nathaniel, 1804-1864",Nathaniel Hawthorne,Nathaniel Hawthorne,1,0,0,0,0,0,0,1,1,historical,domestic_and_genre
3,94,Alexander's Bridge,Civil engineers -- Fiction; London (England) -...,civil engineers -- fiction; london (england) -...,"Cather, Willa, 1873-1947",Willa Cather,Willa Cather,1,0,0,0,0,0,0,0,0,NaN,domestic_only
4,105,Persuasion,England -- Social life and customs -- 19th cen...,england -- social life and customs -- 19th cen...,"Austen, Jane, 1775-1817",Jane Austen,Jane Austen,1,0,0,0,0,0,0,0,0,NaN,domestic_only


In [32]:
## get estimated publication years
wanted_ids = set(
    books_df["gutenberg_id"]
    .astype(int)
    .tolist()
)

print("Looking for:", len(wanted_ids), "Gutenberg IDs")

Looking for: 7307 Gutenberg IDs


In [33]:
wanted_ids = set(
    books_df["gutenberg_id"]
    .astype(int)
    .tolist()
)

print("Looking for:", len(wanted_ids), "Gutenberg IDs")

Looking for: 7307 Gutenberg IDs


In [34]:
publication_rows = []

for row in tqdm(
    temporal_ds,
    desc="Finding publication years"
):
    book_id = int(row["Book_ID"])

    if book_id in wanted_ids:
        publication_rows.append({
            "gutenberg_id": book_id,
            "publication_year_llm_first_last_gemma":
                row["LLM_first_last_gemma"],
        })

Finding publication years: 0it [00:00, ?it/s]

In [35]:
publication_df = pd.DataFrame(publication_rows)

publication_df[
    "publication_year_llm_first_last_gemma"
] = pd.to_numeric(
    publication_df[
        "publication_year_llm_first_last_gemma"
    ],
    errors="coerce"
)

# Negative values such as -1 mean no valid estimate.
publication_df.loc[
    publication_df[
        "publication_year_llm_first_last_gemma"
    ] < 0,
    "publication_year_llm_first_last_gemma"
] = pd.NA

publication_df[
    "publication_year_llm_first_last_gemma"
] = publication_df[
    "publication_year_llm_first_last_gemma"
].astype("Int64")

publication_df = publication_df.drop_duplicates(
    subset="gutenberg_id"
)

print(
    "Publication years found:",
    publication_df[
        "publication_year_llm_first_last_gemma"
    ].notna().sum(),
    "/",
    len(books_df)
)

publication_df.head()

Publication years found: 6862 / 7307


,gutenberg_id,publication_year_llm_first_last_gemma
0,11,1865
1,12,1871
2,15,1851
3,16,1911
4,24,1913


In [37]:
books_with_year_df = books_df.merge(
    publication_df,
    on="gutenberg_id",
    how="left",
    validate="one_to_one",
)

print(books_with_year_df.shape)
books_with_year_df.head()

(7307, 19)


,gutenberg_id,title,subjects,subject_text,author_raw,author_display,author_query,is_domestic,is_detective_mystery,is_science_fiction,is_adventure,is_gothic,is_western,is_fantasy,is_historical,has_major_genre,genre_tags,broad_group,publication_year_llm_first_last_gemma
0,24,O Pioneers!,Nebraska -- Fiction; Historical fiction; Front...,nebraska -- fiction; historical fiction; front...,"Cather, Willa, 1873-1947",Willa Cather,Willa Cather,1,0,0,0,0,0,0,1,1,historical,domestic_and_genre,1913
1,46,A Christmas Carol in Prose; Being a Ghost Stor...,Christmas stories; London (England) -- Fiction...,christmas stories; london (england) -- fiction...,"Dickens, Charles, 1812-1870; Leech, John, 1817...",Charles Dickens,Charles Dickens,1,0,0,0,1,0,0,0,1,gothic,domestic_and_genre,1843
2,77,The House of the Seven Gables,Historical fiction; Domestic fiction; Paranorm...,historical fiction; domestic fiction; paranorm...,"Hawthorne, Nathaniel, 1804-1864",Nathaniel Hawthorne,Nathaniel Hawthorne,1,0,0,0,0,0,0,1,1,historical,domestic_and_genre,1851
3,94,Alexander's Bridge,Civil engineers -- Fiction; London (England) -...,civil engineers -- fiction; london (england) -...,"Cather, Willa, 1873-1947",Willa Cather,Willa Cather,1,0,0,0,0,0,0,0,0,NaN,domestic_only,1920
4,105,Persuasion,England -- Social life and customs -- 19th cen...,england -- social life and customs -- 19th cen...,"Austen, Jane, 1775-1817",Jane Austen,Jane Austen,1,0,0,0,0,0,0,0,0,NaN,domestic_only,1818


In [49]:
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"

HEADERS = {
    "User-Agent": (
        "InteriorityResearch/1.0 "
        "(academic literary metadata project; "
        "contact: your-email@example.com)"
    )
}

BATCH_SIZE = 100
REQUEST_DELAY_SECONDS = 1.0

In [39]:
def make_batches(values, batch_size):
    """Split a list into smaller batches."""
    for start in range(0, len(values), batch_size):
        yield values[start:start + batch_size]

In [40]:
def build_wikidata_query(batch_ids):
    """
    Build one SPARQL query for a batch of Gutenberg IDs.
    """

    id_values = " ".join(
        f'"{int(book_id)}"'
        for book_id in batch_ids
    )

    return f"""
    SELECT DISTINCT
      ?gutenberg_id
      ?work
      ?author
      ?authorLabel
      ?genderLabel
      ?citizenshipLabel
      ?authorDescription
    WHERE {{
      VALUES ?gutenberg_id {{
        {id_values}
      }}

      # P2034 = Project Gutenberg ebook ID
      ?work wdt:P2034 ?gutenberg_id.

      # P50 = author
      OPTIONAL {{
        ?work wdt:P50 ?author.

        OPTIONAL {{
          ?author rdfs:label ?authorLabel.
          FILTER(LANG(?authorLabel) = "en")
        }}

        # P21 = sex or gender
        OPTIONAL {{
          ?author wdt:P21 ?gender.
          ?gender rdfs:label ?genderLabel.
          FILTER(LANG(?genderLabel) = "en")
        }}

        # P27 = country of citizenship
        OPTIONAL {{
          ?author wdt:P27 ?citizenship.
          ?citizenship rdfs:label ?citizenshipLabel.
          FILTER(LANG(?citizenshipLabel) = "en")
        }}

        OPTIONAL {{
          ?author schema:description ?authorDescription.
          FILTER(LANG(?authorDescription) = "en")
        }}
      }}
    }}
    ORDER BY ?gutenberg_id ?authorLabel
    """

In [41]:
def run_wikidata_query(query, max_retries=5):
    """Send one SPARQL request with retry handling."""

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                WIKIDATA_ENDPOINT,
                data={
                    "query": query,
                    "format": "json",
                },
                headers=HEADERS,
                timeout=90,
            )

            if response.status_code in {
                429, 500, 502, 503, 504
            }:
                wait_seconds = attempt * 5

                print(
                    f"Temporary error "
                    f"{response.status_code}. "
                    f"Waiting {wait_seconds} seconds."
                )

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()

            return response.json()[
                "results"
            ]["bindings"]

        except requests.RequestException as error:
            if attempt == max_retries:
                raise RuntimeError(
                    "Wikidata request failed after "
                    f"{max_retries} attempts."
                ) from error

            wait_seconds = attempt * 5

            print(
                f"Request error: {error}. "
                f"Retrying in {wait_seconds} seconds."
            )

            time.sleep(wait_seconds)

    return []

In [42]:
def get_value(binding, field):
    """Safely read one value from a SPARQL result."""
    return binding.get(field, {}).get("value")

In [51]:
def extract_qid(url):
    """Convert a Wikidata URL into a QID."""
    if not url:
        return pd.NA

    return url.rsplit("/", 1)[-1]

In [44]:
def combine_unique(series):
    """
    Combine multiple values into one pipe-separated cell.
    """

    values = sorted({
        str(value).strip()
        for value in series.dropna()
        if str(value).strip()
    })

    if not values:
        return pd.NA

    return " | ".join(values)

In [47]:
all_gutenberg_ids = (
    books_with_year_df["gutenberg_id"]
    .astype(int)
    .drop_duplicates()
    .tolist()
)

batches = list(
    make_batches(
        all_gutenberg_ids,
        100
    )
)

print("Number of batches:", len(batches))

Number of batches: 74


In [52]:
wikidata_rows = []

for batch_number, batch_ids in enumerate(
    tqdm(
        batches,
        desc="Getting Wikidata author metadata"
    ),
    start=1
):
    query = build_wikidata_query(batch_ids)

    bindings = run_wikidata_query(query)

    for binding in bindings:
        wikidata_rows.append({
            "gutenberg_id": get_value(
                binding,
                "gutenberg_id"
            ),
            "wikidata_work_qid": extract_qid(
                get_value(binding, "work")
            ),
            "wikidata_author_qid": extract_qid(
                get_value(binding, "author")
            ),
            "wikidata_author_label": get_value(
                binding,
                "authorLabel"
            ),
            "author_gender": get_value(
                binding,
                "genderLabel"
            ),
            "author_citizenship": get_value(
                binding,
                "citizenshipLabel"
            ),
            "author_description": get_value(
                binding,
                "authorDescription"
            ),
        })

    time.sleep(REQUEST_DELAY_SECONDS)

Getting Wikidata author metadata:   0%|          | 0/74 [00:00<?, ?it/s]

Temporary error 502. Waiting 5 seconds.


In [53]:
wikidata_raw_df = pd.DataFrame(wikidata_rows)

wikidata_raw_df["gutenberg_id"] = pd.to_numeric(
    wikidata_raw_df["gutenberg_id"],
    errors="coerce"
).astype("Int64")

print("Raw Wikidata rows:", len(wikidata_raw_df))

wikidata_raw_df.head()

Raw Wikidata rows: 1157


,gutenberg_id,wikidata_work_qid,wikidata_author_qid,wikidata_author_label,author_gender,author_citizenship,author_description
0,1023,Q883305,Q5686,Charles Dickens,male,United Kingdom of Great Britain and Ireland,English novelist and social critic (1812–1870)
1,1047,Q10381652,Q42511,H. G. Wells,male,United Kingdom of Great Britain and Ireland,English writer (1866–1946)
2,1047,Q10381652,Q42511,H. G. Wells,male,United Kingdom,English writer (1866–1946)
3,105,Q333179,Q36322,Jane Austen,female,United Kingdom of Great Britain and Ireland,English novelist (1775–1817)
4,1093,Q27331,Q170509,Henry James,male,United Kingdom of Great Britain and Ireland,American and British writer (1843–1916)


In [54]:
wikidata_author_df = (
    wikidata_raw_df
    .groupby(
        "gutenberg_id",
        as_index=False,
        dropna=False,
    )
    .agg(
        wikidata_work_qid=(
            "wikidata_work_qid",
            combine_unique,
        ),
        wikidata_author_qid=(
            "wikidata_author_qid",
            combine_unique,
        ),
        wikidata_author_label=(
            "wikidata_author_label",
            combine_unique,
        ),
        author_gender=(
            "author_gender",
            combine_unique,
        ),
        author_citizenship=(
            "author_citizenship",
            combine_unique,
        ),
        author_description=(
            "author_description",
            combine_unique,
        ),
    )
)

print(
    "Unique Gutenberg books matched by Wikidata:",
    wikidata_author_df["gutenberg_id"].nunique()
)

wikidata_author_df.head()

Unique Gutenberg books matched by Wikidata: 848


,gutenberg_id,wikidata_work_qid,wikidata_author_qid,wikidata_author_label,author_gender,author_citizenship,author_description
0,11,Q112864267,Q38082,Lewis Carroll,male,United Kingdom of Great Britain and Ireland,British author and scholar (1832–1898)
1,12,Q220331,Q38082,Lewis Carroll,male,United Kingdom of Great Britain and Ireland,British author and scholar (1832–1898)
2,16,Q3435337,Q81796,J. M. Barrie,male,United Kingdom | United Kingdom of Great Brita...,Scottish writer and playwright (1860–1937)
3,24,Q2008343,Q229480,Willa Cather,female,United States,American writer (1873–1947)
4,32,Q5739278,Q287752,Charlotte Perkins Gilman,female,United States,"American feminist, writer, artist, and lecture..."


In [55]:
final_df = books_with_year_df.merge(
    wikidata_author_df[
        [
            "gutenberg_id",
            "author_gender",
            "author_citizenship",
            "author_description",
        ]
    ],
    on="gutenberg_id",
    how="left",
    validate="one_to_one",
)

In [56]:
print(final_df.columns[-4:].tolist())

['publication_year_llm_first_last_gemma', 'author_gender', 'author_citizenship', 'author_description']


In [57]:
publication_year_found = final_df[
    "publication_year_llm_first_last_gemma"
].notna().sum()

print(
    "Publication years found:",
    publication_year_found,
    "/",
    len(final_df)
)

Publication years found: 6862 / 7307


In [58]:
author_books_found = final_df[
    "author_description"
].notna().sum()

print(
    "Books with Wikidata author metadata:",
    author_books_found,
    "/",
    len(final_df)
)

Books with Wikidata author metadata: 807 / 7307


In [59]:
unmatched_books_df = final_df.loc[
    final_df["author_description"].isna()
].copy()

print(
    "Books missing author metadata:",
    len(unmatched_books_df)
)

Books missing author metadata: 6500


In [60]:
missing_unique_authors = (
    unmatched_books_df["author_query"]
    .dropna()
    .nunique()
)

print(
    "Unique author_query values among missing books:",
    missing_unique_authors
)

Unique author_query values among missing books: 2130


## get author metadata using author name

In [62]:
%pip install -q pandas requests datasets tqdm rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.0 MB/s eta 0:00:00


In [64]:
%run "/content/get_full_metadata_name_match.py"

Loaded publication-year checkpoint: /content/pg_candidate_with_publication_year.csv
Publication years found: 6862 / 7307
Unique author identities to search: 2216
Author identities with at least one lifespan year: 1893
Resuming search checkpoint; completed author identities: 1050


Searching Wikidata by author name:   0%|          | 0/2216 [00:00<?, ?it/s]

Temporary Wikidata error 429; retrying in 3s
Temporary Wikidata error 429; retrying in 6s
Temporary Wikidata error 429; retrying in 9s
Temporary Wikidata error 429; retrying in 3s
Temporary Wikidata error 429; retrying in 6s
Temporary Wikidata error 429; retrying in 3s
Temporary Wikidata error 429; retrying in 6s
Temporary Wikidata error 429; retrying in 3s
Temporary Wikidata error 429; retrying in 6s
Temporary Wikidata error 429; retrying in 9s
Temporary Wikidata error 429; retrying in 3s
Saved name-search candidates: /content/wikidata_author_name_candidates.csv


Fetching Wikidata entities:   0%|          | 0/86 [00:00<?, ?it/s]

Fetching Wikidata entities:   0%|          | 0/2 [00:00<?, ?it/s]

Scoring candidates:   0%|          | 0/4622 [00:00<?, ?it/s]

Saved all scored candidates: /content/wikidata_author_candidates_scored.csv
Saved selected author lookup: /content/wikidata_author_lookup_selected.csv

Saved final dataset: /content/pg_candidate_v1_with_name_matched_metadata.csv
Saved audit dataset: /content/pg_candidate_v1_with_name_matched_metadata_audit.csv
Final shape: (7307, 22)

Missing values in requested columns:
publication_year_llm_first_last_gemma     445
author_gender                            1356
author_citizenship                       1682
author_description                       1353
dtype: int64

Book-level coverage:
publication_year_llm_first_last_gemma: 6862 / 7307 (93.9%)
author_gender: 5951 / 7307 (81.4%)
author_citizenship: 5625 / 7307 (77.0%)
author_description: 5954 / 7307 (81.5%)

Author match statuses:
author_match_status
matched_by_lifespan                5825
unresolved                         1258
matched_by_strong_name              167
needs_review_ambiguous_name          56
needs_review_ambiguous_lifesp

# pg_candidate_v2

In [3]:
%pip install -q pandas requests datasets tqdm rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.4 MB/s eta 0:00:00


In [7]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
from pathlib import Path

RESULT_DIR = Path(
    "/content/drive/MyDrive/Interiority_26Spring/book_metadata_result"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Results will be saved to:", RESULT_DIR)

Results will be saved to: /content/drive/MyDrive/Interiority_26Spring/book_metadata_result


In [9]:
test_path = RESULT_DIR / "test.txt"

test_path.write_text("Drive saving works.", encoding="utf-8")

print(test_path.exists())
print(test_path)

True
/content/drive/MyDrive/Interiority_26Spring/book_metadata_result/test.txt


In [10]:
%run "/content/get_pg_v2_hybrid_metadata.py"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All outputs will be saved to: /content/drive/MyDrive/Interiority_26Spring/book_metadata_result
Books to process: 7388
Unique Gutenberg IDs: 7388
Unique author_display values: 2196


Finding publication years: 0it [00:00, ?it/s]

Saved publication-year checkpoint: /content/drive/MyDrive/Interiority_26Spring/book_metadata_result/v2_full_publication_year_checkpoint.csv
Publication years found: 6943 / 7388


Matching authors by Gutenberg ID:   0%|          | 0/148 [00:00<?, ?it/s]

Books with exact Gutenberg-ID author match: 877 / 7388
Canonical author identities: 2197
Canonical identities seeded by exact Gutenberg match: 349
Canonical author identities requiring name search: 1845
Completed name-search identities from checkpoint: 0


Searching Wikidata author candidates:   0%|          | 0/1845 [00:00<?, ?it/s]

KeyboardInterrupt: 

# pg_candidate_v3

In [3]:
%pip install -q pandas datasets tqdm

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
from google.colab import files


INPUT_CSV = "/content/British Writers - British Books.csv"
OUTPUT_CSV = "/content/British_Books_with_publication_year.csv"


# Load your British Books CSV
books_df = pd.read_csv(INPUT_CSV)

books_df["gutenberg_id"] = pd.to_numeric(
    books_df["gutenberg_id"],
    errors="coerce"
).astype("Int64")

wanted_ids = set(
    books_df["gutenberg_id"]
    .dropna()
    .astype(int)
)

print("Books in input:", len(books_df))
print("Unique Gutenberg IDs:", len(wanted_ids))


# Load the Hugging Face temporal corpus
temporal_ds = load_dataset(
    "Texttechnologylab/project-gutenberg-temporal-corpus",
    split="all",
    streaming=True,
)


# Find the requested Gutenberg IDs
publication_rows = []
found_ids = set()

for row in tqdm(
    temporal_ds,
    desc="Finding publication years"
):
    try:
        book_id = int(row["Book_ID"])
    except (TypeError, ValueError, KeyError):
        continue

    if book_id in wanted_ids and book_id not in found_ids:
        publication_rows.append({
            "gutenberg_id": book_id,
            "LLM_first_last_gemma": row.get(
                "LLM_first_last_gemma"
            ),
        })

        found_ids.add(book_id)

    # Stop once every requested ID has been found
    if len(found_ids) == len(wanted_ids):
        break


# Create the year lookup table
year_df = pd.DataFrame(
    publication_rows,
    columns=[
        "gutenberg_id",
        "LLM_first_last_gemma",
    ]
)

year_df["gutenberg_id"] = pd.to_numeric(
    year_df["gutenberg_id"],
    errors="coerce"
).astype("Int64")

year_df["LLM_first_last_gemma"] = pd.to_numeric(
    year_df["LLM_first_last_gemma"],
    errors="coerce"
)

# Treat negative values such as -1 as missing
year_df.loc[
    year_df["LLM_first_last_gemma"] < 0,
    "LLM_first_last_gemma"
] = pd.NA

year_df["LLM_first_last_gemma"] = year_df[
    "LLM_first_last_gemma"
].astype("Int64")

year_df = year_df.drop_duplicates(
    subset="gutenberg_id"
)


# Append the year column to the original dataset
result_df = books_df.merge(
    year_df,
    on="gutenberg_id",
    how="left",
    validate="many_to_one"
)

result_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# Report coverage
found = result_df[
    "LLM_first_last_gemma"
].notna().sum()

print()
print("Final rows:", len(result_df))
print(
    "Publication years found:",
    f"{found} / {len(result_df)}",
    f"({found / len(result_df):.1%})"
)
print("Saved locally to:", OUTPUT_CSV)

display(
    result_df[
        [
            "gutenberg_id",
            "LLM_first_last_gemma"
        ]
    ].head(10)
)


# Download the completed CSV
files.download(OUTPUT_CSV)

Books in input: 2316
Unique Gutenberg IDs: 2316


README.md:   0%|          | 0.00/2.80k [00:00<?, ?B/s]

Finding publication years: 0it [00:00, ?it/s]


Final rows: 2316
Publication years found: 2151 / 2316 (92.9%)
Saved locally to: /content/British_Books_with_publication_year.csv


,gutenberg_id,LLM_first_last_gemma
0,27,1874
1,35,1895
2,36,1898
3,42,1886
4,43,1886
5,46,1843
6,60,1905
7,82,1820
8,84,1818
9,95,1894


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# updated brisih writers

## get year via hugging face

In [1]:
%pip install -q pandas datasets tqdm

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
from google.colab import files


INPUT_CSV = "/content/British Writers - all books updated.csv"
OUTPUT_CSV = "/content/British_Writers_all_books_with_year.csv"


# Load the updated books dataset
books_df = pd.read_csv(
    INPUT_CSV,
    low_memory=False
)

books_df["gutenberg_id"] = pd.to_numeric(
    books_df["gutenberg_id"],
    errors="coerce"
).astype("Int64")

wanted_ids = set(
    books_df["gutenberg_id"]
    .dropna()
    .astype(int)
)

print("Books in input:", len(books_df))
print("Unique Gutenberg IDs:", len(wanted_ids))


# Load the Hugging Face temporal corpus
temporal_ds = load_dataset(
    "Texttechnologylab/project-gutenberg-temporal-corpus",
    split="all",
    streaming=True,
)


# Retrieve publication-year predictions
publication_rows = []
found_ids = set()

for row in tqdm(
    temporal_ds,
    desc="Finding publication years"
):
    try:
        book_id = int(row["Book_ID"])
    except (TypeError, ValueError, KeyError):
        continue

    if book_id in wanted_ids and book_id not in found_ids:
        publication_rows.append({
            "gutenberg_id": book_id,
            "LLM_first_last_gemma": row.get(
                "LLM_first_last_gemma"
            ),
        })

        found_ids.add(book_id)

    if len(found_ids) == len(wanted_ids):
        break


# Clean the publication-year lookup table
year_df = pd.DataFrame(
    publication_rows,
    columns=[
        "gutenberg_id",
        "LLM_first_last_gemma",
    ]
)

year_df["gutenberg_id"] = pd.to_numeric(
    year_df["gutenberg_id"],
    errors="coerce"
).astype("Int64")

year_df["LLM_first_last_gemma"] = pd.to_numeric(
    year_df["LLM_first_last_gemma"],
    errors="coerce"
)

# Negative values such as -1 are treated as missing
year_df.loc[
    year_df["LLM_first_last_gemma"] < 0,
    "LLM_first_last_gemma"
] = pd.NA

year_df["LLM_first_last_gemma"] = year_df[
    "LLM_first_last_gemma"
].astype("Int64")

year_df = year_df.drop_duplicates(
    subset="gutenberg_id"
)


# Append the publication year
result_df = books_df.merge(
    year_df,
    on="gutenberg_id",
    how="left",
    validate="many_to_one"
)


# Add missing indicator
result_df["publication_year_missing"] = (
    result_df["LLM_first_last_gemma"]
    .isna()
    .map({
        True: "Yes",
        False: "No",
    })
)


# Save locally
result_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# Report coverage
found = result_df[
    "LLM_first_last_gemma"
].notna().sum()

missing = result_df[
    "publication_year_missing"
].eq("Yes").sum()

print()
print("Final rows:", len(result_df))
print(
    "Publication years found:",
    f"{found} / {len(result_df)}",
    f"({found / len(result_df):.1%})"
)
print("Missing publication years:", missing)
print("Saved locally to:", OUTPUT_CSV)

display(
    result_df[
        [
            "gutenberg_id",
            "title",
            "author_display",
            "LLM_first_last_gemma",
            "publication_year_missing",
        ]
    ].head(10)
)


# Download completed CSV
files.download(OUTPUT_CSV)

Books in input: 6618
Unique Gutenberg IDs: 6618


Finding publication years: 0it [00:00, ?it/s]


Final rows: 6618
Publication years found: 6192 / 6618 (93.6%)
Missing publication years: 426
Saved locally to: /content/British_Writers_all_books_with_year.csv


,gutenberg_id,title,author_display,LLM_first_last_gemma,publication_year_missing
0,15,"Moby-Dick; or, The Whale",Herman Melville,1851,No
1,24,O Pioneers!,Willa Cather,1913,No
2,27,Far from the Madding Crowd,Thomas Hardy,1874,No
3,32,Herland,Charlotte Perkins Gilman,1915,No
4,33,The Scarlet Letter,Nathaniel Hawthorne,1850,No
5,35,The Time Machine,H. G. Wells,1895,No
6,36,The war of the worlds,H. G. Wells,1898,No
7,42,The strange case of Dr. Jekyll and Mr. Hyde,Robert Louis Stevenson,1886,No
8,43,The strange case of Dr. Jekyll and Mr. Hyde,Robert Louis Stevenson,1886,No
9,46,A Christmas Carol in Prose; Being a Ghost Stor...,Charles Dickens,1843,No


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## fill missing years via wikipedia/wikidata or open library

In [2]:
%pip install -q pandas requests tqdm rapidfuzz

import re
import time
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
from rapidfuzz.fuzz import ratio
from tqdm.auto import tqdm
from google.colab import files


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_CSV = "/content/British_Writers_all_books_with_year.csv"

OUTPUT_CSV = (
    "/content/British_Writers_all_books_with_researched_years.csv"
)

CHECKPOINT_CSV = (
    "/content/publication_year_research_checkpoint.csv"
)

CONTACT_EMAIL = "YOUR_REAL_EMAIL@example.com"

REQUEST_DELAY = 0.6
SAVE_EVERY = 20


# ============================================================
# LOAD DATA
# ============================================================

books_df = pd.read_csv(INPUT_CSV, low_memory=False)

required_columns = {
    "gutenberg_id",
    "title",
    "LLM_first_last_gemma",
    "publication_year_missing",
}

missing_columns = required_columns - set(books_df.columns)

if missing_columns:
    raise ValueError(
        f"Input CSV is missing columns: {sorted(missing_columns)}"
    )


# Automatically choose the best author column available
author_column_candidates = [
    "author_display",
    "author_query",
    "author",
    "author_raw",
]

AUTHOR_COLUMN = next(
    (
        col
        for col in author_column_candidates
        if col in books_df.columns
    ),
    None,
)

if AUTHOR_COLUMN is None:
    raise ValueError(
        "Could not find an author column. "
        "Expected author_display, author_query, author, or author_raw."
    )

print("Using author column:", AUTHOR_COLUMN)
print("Total rows:", len(books_df))

missing_mask = (
    books_df["publication_year_missing"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("yes")
)

print("Books requiring research:", missing_mask.sum())


# ============================================================
# HELPERS
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "InteriorityPublicationYearResearch/1.0 "
        f"(academic research; contact: {CONTACT_EMAIL})"
    )
})


def clean_text(value):
    if pd.isna(value):
        return ""

    text = str(value)
    text = re.sub(r"\[[^\]]*\]", " ", text)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def normalize_text(value):
    text = clean_text(value).casefold()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def clean_author_for_search(value):
    """
    Remove Gutenberg role annotations and lifespan years while
    keeping the main author's name.
    """

    text = clean_text(value)

    # Keep only the first listed person
    text = text.split(";")[0]

    # Remove lifespan-like values
    text = re.sub(
        r"\b\d{3,4}\??\s*[-–—]\s*\d{0,4}\??\b",
        " ",
        text,
    )

    text = re.sub(r"\b\d{3,4}\??\b", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip(" ,.;")


def extract_year_from_wikidata_time(value):
    """
    Convert values such as +1813-01-28T00:00:00Z into 1813.
    """

    if not value:
        return None

    match = re.search(r"([+-])(\d{3,4})-", str(value))

    if not match:
        return None

    year = int(match.group(2))

    if match.group(1) == "-":
        year = -year

    return year


def valid_publication_year(year):
    if year is None:
        return False

    # Adjust later if the corpus needs a different range
    return 1000 <= int(year) <= 2026


def request_json(url, params, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(
                url,
                params=params,
                timeout=40,
            )

            if response.status_code == 429:
                retry_after = response.headers.get(
                    "Retry-After"
                )

                wait = (
                    int(retry_after)
                    if retry_after
                    and str(retry_after).isdigit()
                    else 5 * attempt
                )

                print(
                    f"Rate limited; waiting {wait} seconds."
                )
                time.sleep(wait)
                continue

            if response.status_code in {
                500, 502, 503, 504
            }:
                wait = 3 * attempt

                print(
                    f"Temporary error "
                    f"{response.status_code}; "
                    f"waiting {wait} seconds."
                )

                time.sleep(wait)
                continue

            response.raise_for_status()
            return response.json()

        except requests.RequestException as error:
            if attempt == max_retries:
                return None

            time.sleep(3 * attempt)

    return None


# ============================================================
# METHOD 1: WIKIPEDIA SEARCH -> WIKIDATA P577
# ============================================================

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
WIKIDATA_API = "https://www.wikidata.org/w/api.php"


def search_wikipedia_candidates(title, author):
    query = f'"{title}" "{author}"'

    data = request_json(
        WIKIPEDIA_API,
        {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srnamespace": 0,
            "srlimit": 5,
            "format": "json",
        },
    )

    if not data:
        return []

    results = data.get(
        "query",
        {},
    ).get(
        "search",
        [],
    )

    if not results:
        # Retry with a less strict query
        data = request_json(
            WIKIPEDIA_API,
            {
                "action": "query",
                "list": "search",
                "srsearch": f"{title} {author}",
                "srnamespace": 0,
                "srlimit": 5,
                "format": "json",
            },
        )

        if not data:
            return []

        results = data.get(
            "query",
            {},
        ).get(
            "search",
            [],
        )

    return results


def get_wikipedia_page_details(pageids):
    if not pageids:
        return {}

    data = request_json(
        WIKIPEDIA_API,
        {
            "action": "query",
            "pageids": "|".join(map(str, pageids)),
            "prop": "pageprops|extracts",
            "ppprop": "wikibase_item",
            "exintro": 1,
            "explaintext": 1,
            "format": "json",
        },
    )

    if not data:
        return {}

    return data.get(
        "query",
        {},
    ).get(
        "pages",
        {},
    )


def get_wikidata_publication_year(qid):
    if not qid:
        return None

    data = request_json(
        WIKIDATA_API,
        {
            "action": "wbgetentities",
            "ids": qid,
            "props": "claims",
            "format": "json",
        },
    )

    if not data:
        return None

    entity = data.get(
        "entities",
        {},
    ).get(
        qid,
        {},
    )

    claims = entity.get(
        "claims",
        {},
    )

    publication_claims = claims.get(
        "P577",
        [],
    )

    years = []

    for claim in publication_claims:
        try:
            time_value = (
                claim["mainsnak"]
                ["datavalue"]
                ["value"]
                ["time"]
            )

            year = extract_year_from_wikidata_time(
                time_value
            )

            if valid_publication_year(year):
                years.append(year)

        except (
            KeyError,
            TypeError,
        ):
            continue

    if not years:
        return None

    # When several publication dates exist, use the earliest
    return min(years)


def wikipedia_publication_year(title, author):
    candidates = search_wikipedia_candidates(
        title,
        author,
    )

    if not candidates:
        return None

    pageids = [
        item["pageid"]
        for item in candidates
        if "pageid" in item
    ]

    pages = get_wikipedia_page_details(pageids)

    target_title = normalize_text(title)
    target_author = normalize_text(author)

    scored = []

    for candidate in candidates:
        pageid = str(candidate.get("pageid"))
        page = pages.get(pageid, {})

        page_title = page.get(
            "title",
            candidate.get("title", ""),
        )

        extract = page.get("extract", "")

        title_score = ratio(
            target_title,
            normalize_text(page_title),
        )

        author_tokens = [
            token
            for token in target_author.split()
            if len(token) >= 3
        ]

        extract_normalized = normalize_text(extract)

        author_hits = sum(
            token in extract_normalized
            for token in author_tokens
        )

        author_ratio = (
            author_hits / len(author_tokens)
            if author_tokens
            else 0
        )

        qid = (
            page.get("pageprops", {})
            .get("wikibase_item")
        )

        score = (
            0.75 * title_score
            + 25 * author_ratio
        )

        scored.append({
            "page_title": page_title,
            "pageid": pageid,
            "qid": qid,
            "score": score,
            "title_score": title_score,
            "author_ratio": author_ratio,
        })

    scored.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    for candidate in scored:
        # Conservative acceptance threshold
        if (
            candidate["title_score"] < 82
            or candidate["author_ratio"] < 0.40
        ):
            continue

        year = get_wikidata_publication_year(
            candidate["qid"]
        )

        if valid_publication_year(year):
            confidence = (
                "high"
                if (
                    candidate["title_score"] >= 92
                    and candidate["author_ratio"] >= 0.60
                )
                else "medium"
            )

            return {
                "year": int(year),
                "source": "Wikipedia/Wikidata P577",
                "source_url": (
                    "https://en.wikipedia.org/wiki/"
                    + quote(
                        candidate["page_title"]
                        .replace(" ", "_")
                    )
                ),
                "confidence": confidence,
                "match_score": round(
                    candidate["score"],
                    2,
                ),
                "matched_title": candidate[
                    "page_title"
                ],
            }

    return None


# ============================================================
# METHOD 2: OPEN LIBRARY FALLBACK
# ============================================================

OPEN_LIBRARY_API = "https://openlibrary.org/search.json"


def open_library_publication_year(title, author):
    data = request_json(
        OPEN_LIBRARY_API,
        {
            "title": title,
            "author": author,
            "fields": (
                "key,title,author_name,"
                "first_publish_year"
            ),
            "limit": 10,
        },
    )

    if not data:
        return None

    documents = data.get("docs", [])

    target_title = normalize_text(title)
    target_author = normalize_text(author)

    scored = []

    for doc in documents:
        year = doc.get("first_publish_year")

        if not valid_publication_year(year):
            continue

        returned_title = doc.get("title", "")
        returned_authors = doc.get(
            "author_name",
            [],
        )

        title_score = ratio(
            target_title,
            normalize_text(returned_title),
        )

        author_score = max(
            [
                ratio(
                    target_author,
                    normalize_text(name),
                )
                for name in returned_authors
            ]
            or [0]
        )

        total_score = (
            0.70 * title_score
            + 0.30 * author_score
        )

        scored.append({
            "year": int(year),
            "title": returned_title,
            "title_score": title_score,
            "author_score": author_score,
            "score": total_score,
            "key": doc.get("key"),
        })

    if not scored:
        return None

    scored.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    best = scored[0]

    # Require both the title and author to be reasonably close
    if (
        best["title_score"] < 85
        or best["author_score"] < 75
    ):
        return None

    confidence = (
        "high"
        if (
            best["title_score"] >= 94
            and best["author_score"] >= 90
        )
        else "medium"
    )

    source_url = (
        "https://openlibrary.org"
        + str(best["key"])
        if best["key"]
        else ""
    )

    return {
        "year": best["year"],
        "source": "Open Library first_publish_year",
        "source_url": source_url,
        "confidence": confidence,
        "match_score": round(
            best["score"],
            2,
        ),
        "matched_title": best["title"],
    }


# ============================================================
# LOAD OR CREATE CHECKPOINT
# ============================================================

checkpoint_path = Path(CHECKPOINT_CSV)

if checkpoint_path.exists():
    checkpoint_df = pd.read_csv(
        checkpoint_path,
        low_memory=False,
    )

    completed_ids = set(
        pd.to_numeric(
            checkpoint_df["gutenberg_id"],
            errors="coerce",
        )
        .dropna()
        .astype(int)
    )

    research_rows = checkpoint_df.to_dict(
        orient="records"
    )

    print(
        "Resuming checkpoint. Completed:",
        len(completed_ids),
    )

else:
    completed_ids = set()
    research_rows = []


# ============================================================
# RESEARCH MISSING YEARS
# ============================================================

missing_books_df = books_df.loc[
    missing_mask
].copy()

rows_to_process = missing_books_df.loc[
    ~missing_books_df["gutenberg_id"]
    .astype("Int64")
    .isin(completed_ids)
]

print(
    "Remaining books to research:",
    len(rows_to_process),
)


for counter, (_, row) in enumerate(
    tqdm(
        rows_to_process.iterrows(),
        total=len(rows_to_process),
        desc="Researching publication years",
    ),
    start=1,
):
    gutenberg_id = row["gutenberg_id"]
    title = clean_text(row["title"])

    author = clean_author_for_search(
        row[AUTHOR_COLUMN]
    )

    result = None

    # First try Wikipedia
    if title and author:
        result = wikipedia_publication_year(
            title,
            author,
        )

        time.sleep(REQUEST_DELAY)

    # Then try Open Library
    if result is None and title and author:
        result = open_library_publication_year(
            title,
            author,
        )

        time.sleep(REQUEST_DELAY)

    if result:
        research_rows.append({
            "gutenberg_id": gutenberg_id,
            "researched_publication_year":
                result["year"],
            "publication_year_research_source":
                result["source"],
            "publication_year_source_url":
                result["source_url"],
            "publication_year_confidence":
                result["confidence"],
            "publication_year_match_score":
                result["match_score"],
            "publication_year_matched_title":
                result["matched_title"],
            "publication_year_research_status":
                "found",
        })

    else:
        research_rows.append({
            "gutenberg_id": gutenberg_id,
            "researched_publication_year":
                pd.NA,
            "publication_year_research_source":
                pd.NA,
            "publication_year_source_url":
                pd.NA,
            "publication_year_confidence":
                pd.NA,
            "publication_year_match_score":
                pd.NA,
            "publication_year_matched_title":
                pd.NA,
            "publication_year_research_status":
                "not_found",
        })

    if counter % SAVE_EVERY == 0:
        pd.DataFrame(
            research_rows
        ).to_csv(
            CHECKPOINT_CSV,
            index=False,
            encoding="utf-8-sig",
        )


# Save final checkpoint
research_df = pd.DataFrame(research_rows)

research_df.to_csv(
    CHECKPOINT_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# MERGE BACK INTO ORIGINAL DATA
# ============================================================

research_df["gutenberg_id"] = pd.to_numeric(
    research_df["gutenberg_id"],
    errors="coerce",
).astype("Int64")

research_df[
    "researched_publication_year"
] = pd.to_numeric(
    research_df["researched_publication_year"],
    errors="coerce",
).astype("Int64")


result_df = books_df.merge(
    research_df,
    on="gutenberg_id",
    how="left",
    validate="many_to_one",
)


# Use Hugging Face year first, researched year second
result_df["publication_year_final"] = (
    result_df["LLM_first_last_gemma"]
    .fillna(
        result_df[
            "researched_publication_year"
        ]
    )
    .astype("Int64")
)


result_df["publication_year_final_source"] = (
    result_df[
        "publication_year_research_source"
    ]
)

result_df.loc[
    result_df["LLM_first_last_gemma"].notna(),
    "publication_year_final_source"
] = "Hugging Face LLM_first_last_gemma"


result_df["publication_year_still_missing"] = (
    result_df["publication_year_final"]
    .isna()
    .map({
        True: "Yes",
        False: "No",
    })
)


result_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# REPORT RESULTS
# ============================================================

original_missing = missing_mask.sum()

newly_found = (
    result_df["researched_publication_year"]
    .notna()
    .sum()
)

still_missing = (
    result_df["publication_year_final"]
    .isna()
    .sum()
)

print()
print("Originally missing:", original_missing)
print("Newly found:", newly_found)
print("Still missing:", still_missing)

print()
print("Research sources:")
print(
    result_df[
        "publication_year_research_source"
    ]
    .value_counts(dropna=False)
)

print()
print("Confidence levels:")
print(
    result_df[
        "publication_year_confidence"
    ]
    .value_counts(dropna=False)
)

print()
print("Saved to:", OUTPUT_CSV)


display(
    result_df.loc[
        result_df[
            "researched_publication_year"
        ].notna(),
        [
            "gutenberg_id",
            "title",
            AUTHOR_COLUMN,
            "researched_publication_year",
            "publication_year_research_source",
            "publication_year_confidence",
            "publication_year_source_url",
        ],
    ].head(20)
)


files.download(OUTPUT_CSV)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.8 MB/s eta 0:00:00
Using author column: author_display
Total rows: 6618
Books requiring research: 426
Remaining books to research: 426


Researching publication years:   0%|          | 0/426 [00:00<?, ?it/s]


Originally missing: 426
Newly found: 267
Still missing: 159

Research sources:
publication_year_research_source
NaN                                6192
Open Library first_publish_year     204
<NA>                                159
Wikipedia/Wikidata P577              63
Name: count, dtype: int64

Confidence levels:
publication_year_confidence
NaN       6192
high       250
<NA>       159
medium      17
Name: count, dtype: int64

Saved to: /content/British_Writers_all_books_with_researched_years.csv


,gutenberg_id,title,author_display,researched_publication_year,publication_year_research_source,publication_year_confidence,publication_year_source_url
1993,21241,The Rifle Rangers,Mayne Reid,1868,Open Library first_publish_year,high,https://openlibrary.org/works/OL5305933W
3180,36050,Rodney Stone,Arthur Conan Doyle,1896,Wikipedia/Wikidata P577,high,https://en.wikipedia.org/wiki/Rodney_Stone
6196,74409,Blowing weather,John T. McIntyre,1923,Open Library first_publish_year,high,https://openlibrary.org/works/OL43015981W
6197,74419,Reuben Sachs,Amy Levy,1888,Open Library first_publish_year,high,https://openlibrary.org/works/OL4313508W
6198,74437,Those barren leaves,Aldous Huxley,1925,Wikipedia/Wikidata P577,high,https://en.wikipedia.org/wiki/Those_Barren_Leaves
6206,74598,A marrying man,G. B. Stern,1918,Open Library first_publish_year,high,https://openlibrary.org/works/OL1940824W
6207,74602,With George Washington into the wilderness,Edwin L. Sabin,1924,Open Library first_publish_year,high,https://openlibrary.org/works/OL3750817W
6208,74612,Diverging roads,Rose Wilder Lane,1919,Open Library first_publish_year,high,https://openlibrary.org/works/OL2709972W
6213,74687,Captains of souls,Edgar Wallace,1922,Open Library first_publish_year,high,https://openlibrary.org/works/OL1517976W
6214,74690,Cole of Spyglass Mountain,Arthur Preston Hankins,1923,Open Library first_publish_year,high,https://openlibrary.org/works/OL22103891W


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>